# Notebook 07 — Create the basket dataset

> The old timestamp rule excluded 16.6% of purchase events. We now preserve simultaneous events together in unordered time blocks. This notebook creates the prepared basket dataset; training splits and training-only filtering come next.

A **time block** contains all events from one visitor at one exact timestamp. It may contain views, cart actions, or purchases. We call a multi-item block a **transaction basket** only when all its rows are purchases with one shared, nonblank transaction ID.

Run one cell at a time. The expected result is **2,755,641 events**, including **22,457 transaction events**, organized into **2,753,015 time blocks**. These are expected checks for this source version, not results from a run of this notebook.

This changes the target definition: a future target block may contain several correct items. The representation for SASRec/TiSASRec and the multi-positive evaluation rules still need design.

## 1. Find the inputs and choose a fresh output folder

We start from the original CSV and compare our duplicate removal with Notebook 01's saved exact-deduplicated file. Change INPUT_RUN_TAG if you used another name.

The automatic run name prevents older outputs from being overwritten. A manifest starts with status **in_progress** and changes only after all checks and exports finish. If a cell fails, fix the cause and restart with a fresh run.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import time

import pandas as pd
from IPython.display import display

STARTED = time.perf_counter()
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dataset/events.csv").exists()),
    None,
)
assert PROJECT_ROOT is not None, "Open this notebook inside the Recommendation project."
INPUT_RUN_TAG = "notebook_run_2026_10_13"
RAW_FILE = PROJECT_ROOT / "dataset/events.csv"
PREVIOUS_FILE = PROJECT_ROOT / "data/interim" / INPUT_RUN_TAG / "events_exact_deduplicated.csv"
NOTEBOOK_FILE = PROJECT_ROOT / "notebooks/07_build_basket_dataset.ipynb"
CONFIG_FILE = PROJECT_ROOT / "configs/datasets.json"
for path in [RAW_FILE, PREVIOUS_FILE, NOTEBOOK_FILE, CONFIG_FILE]:
    assert path.is_file(), f"Missing input: {path}"

RUN_TAG = "basket_prepared_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
DATA_DIR = PROJECT_ROOT / "data/interim" / RUN_TAG
REPORT_DIR = PROJECT_ROOT / "reports" / RUN_TAG
assert not DATA_DIR.exists() and not REPORT_DIR.exists()
DATA_DIR.mkdir(parents=True, exist_ok=False)
REPORT_DIR.mkdir(parents=True, exist_ok=False)
MANIFEST_FILE = REPORT_DIR / "manifest.json"

FIELDS = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
KEYS = ["visitorid", "timestamp"]
EVENT_TYPES = ["view", "addtocart", "transaction"]
DTYPES = {"timestamp": "int64", "visitorid": "int64", "itemid": "int64",
          "event": "string", "transactionid": "Int64"}

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_checksums = {str(p.relative_to(PROJECT_ROOT)): file_sha256(p)
                   for p in [RAW_FILE, PREVIOUS_FILE, NOTEBOOK_FILE, CONFIG_FILE]}
manifest = {
    "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "in_progress",
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "input_sha256": input_checksums,
    "settings": {
        "eligible_events": EVENT_TYPES, "exact_duplicates": "keep_first_raw_source_row",
        "block_key": KEYS, "block_id_format": "v{visitorid}_t{timestamp}",
        "within_block_order": "unordered",
        "item_membership": "distinct itemid per block",
        "minimum_interaction_filter_applied": False,
        "split_applied": False, "model_representation": "not yet decided",
        "seed": None, "hardware_used": "CPU; no GPU training",
    },
    "data_directory": str(DATA_DIR), "report_directory": str(REPORT_DIR),
    "outputs": [],
}
MANIFEST_FILE.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Prepared dataset:", DATA_DIR)
print("Audits:", REPORT_DIR)

Prepared dataset: E:\ALL MCP Servers\Recommendation\data\interim\basket_prepared_20261013_180506_369663
Audits: E:\ALL MCP Servers\Recommendation\reports\basket_prepared_20261013_180506_369663


## 2. Remove exact copies and preserve source links

**drop_duplicates** compares all five original fields and retains the first copy. The source-row number is the original CSV line number, with the header at line 1. It helps you find an event again; it does not establish an order inside a time block.

Different actions on the same item remain separate event rows.

In [2]:
raw = pd.read_csv(RAW_FILE, dtype=DTYPES)
assert raw.columns.tolist() == FIELDS
assert raw[["timestamp", "visitorid", "event", "itemid"]].notna().all().all()
assert set(raw["event"]) == set(EVENT_TYPES)
assert raw.loc[raw["event"].eq("transaction"), "transactionid"].notna().all()

clean = raw.drop_duplicates(FIELDS, keep="first").copy()
clean["source_row_number"] = clean.index + 2
previous = pd.read_csv(PREVIOUS_FILE, dtype=DTYPES)
pd.testing.assert_frame_equal(clean[FIELDS].reset_index(drop=True), previous[FIELDS].reset_index(drop=True))
del previous
assert len(raw) == 2_756_101
assert len(clean) == 2_755_641
assert clean["event"].eq("transaction").sum() == 22_457
assert not clean.duplicated(FIELDS).any()
print(f"Original events: {len(raw):,}; extra exact copies removed: {len(raw) - len(clean):,}")

Original events: 2,756,101; extra exact copies removed: 460


## 3. Build the three linked tables

The function below returns:
- **events_with_blocks**: every retained event, its source row, and its block ID;
- **block_items**: each distinct item in each block, listed once;
- **block_summary**: the visitor, timestamp, UTC date/time, and counts for each block.

For example, viewing and adding item 42 to a cart at the same time creates two event rows but one block–item membership.

**groupby** gathers events sharing visitor and timestamp. **agg** counts them. The text ID uses that same pair, so shuffling source rows cannot change block identities.

Sorting item IDs makes files reproducible. It is only storage/display order; the model must treat items inside a block as an unordered set.

In [3]:
def build_basket_tables(source_events):
    if source_events.empty:
        raise ValueError("No events to prepare.")
    required = FIELDS + ["source_row_number"]
    if not set(required).issubset(source_events.columns):
        raise ValueError("Missing event fields or source-row numbers.")
    if source_events[KEYS + ["itemid", "event"]].isna().any().any():
        raise ValueError("Missing required block fields.")
    if not set(source_events["event"]).issubset(EVENT_TYPES):
        raise ValueError("Unexpected event type.")

    events = source_events[required].copy()
    events["block_id"] = (
        "v" + events["visitorid"].astype("string")
        + "_t" + events["timestamp"].astype("string")
    )
    # Storage order only: never interpret itemid or source row as event timing.
    events = events.sort_values(
        KEYS + ["itemid", "event", "source_row_number"], kind="stable"
    ).reset_index(drop=True)
    members = events[["block_id", "itemid"]].drop_duplicates().reset_index(drop=True)

    counts_input = events.assign(
        view_count=events["event"].eq("view").astype("int8"),
        addtocart_count=events["event"].eq("addtocart").astype("int8"),
        transaction_count=events["event"].eq("transaction").astype("int8"),
    )
    blocks = counts_input.groupby(KEYS, as_index=False, sort=True).agg(
        block_id=("block_id", "first"),
        event_count=("itemid", "size"),
        distinct_item_count=("itemid", "nunique"),
        view_count=("view_count", "sum"),
        addtocart_count=("addtocart_count", "sum"),
        transaction_count=("transaction_count", "sum"),
        transaction_id_count=("transactionid", "count"),
        distinct_transaction_ids=("transactionid", "nunique"),
    )
    blocks["datetime_utc"] = pd.to_datetime(blocks["timestamp"], unit="ms", utc=True)
    blocks["block_type"] = "other_multi_item_block"
    same_item = blocks["event_count"].gt(1) & blocks["distinct_item_count"].eq(1)
    all_purchases = blocks["transaction_count"].eq(blocks["event_count"])
    valid_id = (blocks["transaction_id_count"].eq(blocks["event_count"])
                & blocks["distinct_transaction_ids"].eq(1))
    multi_item = blocks["distinct_item_count"].gt(1)
    blocks.loc[all_purchases & multi_item & ~valid_id, "block_type"] = "ambiguous_transaction_block"
    blocks.loc[all_purchases & multi_item & valid_id, "block_type"] = "transaction_basket"
    blocks.loc[same_item, "block_type"] = "same_item_tie"
    blocks.loc[blocks["event_count"].eq(1), "block_type"] = "single_event_block"
    blocks = blocks[[
        "block_id", "visitorid", "timestamp", "datetime_utc", "event_count",
        "distinct_item_count", "view_count", "addtocart_count", "transaction_count",
        "transaction_id_count", "distinct_transaction_ids", "block_type",
    ]]
    return events, members, blocks

## 4. Check the grouping on a tiny example

These small checks cover different items at the same time, two actions on the same item, two visitors at the same time, and an item revisited later. They also verify that missing/conflicting transaction IDs do not create a confirmed purchase basket.

The shuffled-input test compares outputs exactly. It checks that file order does not become a fake clock.

In [4]:
toy = pd.DataFrame([
    [1000, 1, "transaction", 10, 99],
    [1000, 1, "transaction", 11, 99],
    [2000, 1, "view", 10, None],
    [2000, 1, "addtocart", 10, None],
    [1000, 2, "view", 10, None],
    [3000, 1, "view", 10, None],
    [4000, 1, "view", 12, None],
    [4000, 1, "view", 13, None],
    [5000, 1, "transaction", 14, 100],
    [5000, 1, "transaction", 15, 101],
    [6000, 1, "transaction", 16, None],
    [6000, 1, "transaction", 17, None],
], columns=FIELDS).astype(DTYPES)
toy["source_row_number"] = range(2, len(toy) + 2)
toy_events, toy_members, toy_blocks = build_basket_tables(toy)
toy_lookup = toy_blocks.set_index("block_id")
assert len(toy_events) == 12 and len(toy_members) == 11 and len(toy_blocks) == 7
assert toy_lookup.loc["v1_t1000", "block_type"] == "transaction_basket"
assert toy_lookup.loc["v1_t2000", "distinct_item_count"] == 1
assert toy_lookup.loc["v1_t2000", "event_count"] == 2
assert toy_lookup.loc["v1_t5000", "block_type"] == "ambiguous_transaction_block"
assert toy_lookup.loc["v1_t6000", "block_type"] == "ambiguous_transaction_block"
assert "v2_t1000" in toy_lookup.index and "v1_t3000" in toy_lookup.index
shuffled = build_basket_tables(toy.sample(frac=1, random_state=7))
for original, reordered in zip((toy_events, toy_members, toy_blocks), shuffled):
    pd.testing.assert_frame_equal(original, reordered)
print("Small-example checks passed.")

Small-example checks passed.


## 5. Build and validate the full prepared dataset

This cell does the main work. A membership is an item belonging to a time block. Assertions stop the notebook if events disappear, block counts do not match, or a source-row link is wrong.

The minimum-five rule is deliberately deferred until training-period eligibility is defined. Retaining an event here does not promise it will be eligible for every later experiment.

In [5]:
events_with_blocks, block_items, block_summary = build_basket_tables(clean)
assert len(events_with_blocks) == 2_755_641
assert len(block_summary) == 2_753_015
assert events_with_blocks["event"].eq("transaction").sum() == 22_457
assert not block_summary["block_id"].duplicated().any()
assert not block_summary.duplicated(KEYS).any()
assert not block_items.duplicated(["block_id", "itemid"]).any()
assert events_with_blocks["block_id"].notna().all()
assert events_with_blocks["block_id"].isin(block_summary["block_id"]).all()
assert events_with_blocks.groupby("block_id")[KEYS].nunique().eq(1).all().all()

summary_index = block_summary.set_index("block_id")
pd.testing.assert_series_equal(
    events_with_blocks.groupby("block_id").size().sort_index(),
    summary_index["event_count"].sort_index(), check_names=False,
)
pd.testing.assert_series_equal(
    block_items.groupby("block_id").size().sort_index(),
    summary_index["distinct_item_count"].sort_index(), check_names=False,
)
assert block_summary["event_count"].sum() == len(events_with_blocks)
assert block_summary["distinct_item_count"].sum() == len(block_items)
assert block_summary[["view_count", "addtocart_count", "transaction_count"]].sum(axis=1).eq(
    block_summary["event_count"]
).all()

tied_blocks = block_summary.loc[block_summary["event_count"].gt(1)]
assert len(tied_blocks) == 2_143 and tied_blocks["event_count"].sum() == 4_769
assert not events_with_blocks["source_row_number"].duplicated().any()
positions = events_with_blocks["source_row_number"].to_numpy() - 2
assert (positions >= 0).all() and (positions < len(raw)).all()
pd.testing.assert_frame_equal(
    events_with_blocks[FIELDS].reset_index(drop=True),
    raw.iloc[positions][FIELDS].reset_index(drop=True),
)
print("Full dataset checks passed. Outputs are ready to export.")
display(block_summary.head())

Full dataset checks passed. Outputs are ready to export.


,block_id,visitorid,timestamp,datetime_utc,event_count,distinct_item_count,view_count,addtocart_count,transaction_count,transaction_id_count,distinct_transaction_ids,block_type
0,v0_t1442004589439,0,1442004589439,2015-09-11 20:49:49.439000+00:00,1,1,1,0,0,0,0,single_event_block
1,v0_t1442004759591,0,1442004759591,2015-09-11 20:52:39.591000+00:00,1,1,1,0,0,0,0,single_event_block
2,v0_t1442004917175,0,1442004917175,2015-09-11 20:55:17.175000+00:00,1,1,1,0,0,0,0,single_event_block
3,v1_t1439487966444,1,1439487966444,2015-08-13 17:46:06.444000+00:00,1,1,1,0,0,0,0,single_event_block
4,v2_t1438969904567,2,1438969904567,2015-08-07 17:51:44.567000+00:00,1,1,1,0,0,0,0,single_event_block


## 6. Prepare count reports and review examples

The count report describes raw events and prepared events. The block-size table counts how many blocks have each combination of event count and distinct-item count.

A **merge** joins block details onto audit event rows using block_id. Its many-to-one check ensures an event cannot accidentally be multiplied by duplicate summary rows. Example files contain complete blocks, so you can count them in Excel.

In [7]:
def population_counts(name, frame):
    return {
        "stage": name, "events": len(frame), "visitors": frame["visitorid"].nunique(),
        "items": frame["itemid"].nunique(), "blocks": len(frame[KEYS].drop_duplicates()),
        "transactions": int(frame["event"].eq("transaction").sum()),
    }

count_report = pd.DataFrame([
    population_counts("raw", raw), population_counts("prepared", events_with_blocks),
])
block_size_distribution = (
    block_summary.groupby(["event_count", "distinct_item_count"], as_index=False)
    .size().rename(columns={"size": "blocks"})
)
audit_fields = ["block_id", "source_row_number", *FIELDS,
                "event_count", "distinct_item_count", "block_type"]
audit_events = events_with_blocks.loc[events_with_blocks["block_id"].isin(tied_blocks["block_id"])]
all_tied_events = audit_events.merge(
    tied_blocks[["block_id", "event_count", "distinct_item_count", "block_type"]],
    on="block_id", validate="many_to_one",
)[audit_fields]
assert len(all_tied_events) == 4_769

# Select up to three whole blocks of each type. These are examples, not a random sample.
example_blocks = block_summary.groupby("block_type", sort=True).head(3)
examples = events_with_blocks.loc[events_with_blocks["block_id"].isin(example_blocks["block_id"])].merge(
    example_blocks[["block_id", "event_count", "distinct_item_count", "block_type"]],
    on="block_id", validate="many_to_one",
)[audit_fields]
assert examples.groupby("block_id").size().eq(
    example_blocks.set_index("block_id")["event_count"]
).all()
display(count_report)
display(block_summary["block_type"].value_counts().rename("blocks"))
display(examples.head(12))

,stage,events,visitors,items,blocks,transactions
0,raw,2756101,1407580,235061,2753015,22457
1,prepared,2755641,1407580,235061,2753015,22457


block_type
single_event_block        2750872
transaction_basket           1633
other_multi_item_block        499
same_item_tie                  11
Name: blocks, dtype: int64

,block_id,source_row_number,timestamp,visitorid,event,itemid,transactionid,event_count,distinct_item_count,block_type
0,v0_t1442004589439,1361689,1442004589439,0,view,285930,<NA>,1,1,single_event_block
1,v0_t1442004759591,1367214,1442004759591,0,view,357564,<NA>,1,1,single_event_block
2,v0_t1442004917175,1367344,1442004917175,0,view,67045,<NA>,1,1,single_event_block
3,v2799_t1440099361419,964037,1440099361419,2799,view,332761,<NA>,2,2,other_multi_item_block
4,v2799_t1440099361419,968313,1440099361419,2799,view,389853,<NA>,2,2,other_multi_item_block
5,v3465_t1434418664698,268761,1434418664698,3465,transaction,8523,11600,3,3,transaction_basket
6,v3465_t1434418664698,280708,1434418664698,3465,transaction,114485,11600,3,3,transaction_basket
7,v3465_t1434418664698,272604,1434418664698,3465,transaction,434048,11600,3,3,transaction_basket
8,v5703_t1438030995619,2649833,1438030995619,5703,transaction,186214,3297,2,2,transaction_basket
9,v5703_t1438030995619,2664130,1438030995619,5703,transaction,295495,3297,2,2,transaction_basket


## 7. Export into this run's new folders

The three main CSVs have more rows than one Excel worksheet can hold. The manifest labels that limitation and lists each output's size and row count.

Use the small audit CSVs for ordinary Excel inspection. For full files, use pandas or Excel Power Query with a filtered query; do not open the complete CSV directly and assume Excel loaded every row.

The manifest is marked complete only after all files are written and the two input data checksums still match.

In [8]:
exports = [
    (DATA_DIR / "events_with_blocks.csv", events_with_blocks),
    (DATA_DIR / "block_items.csv", block_items),
    (DATA_DIR / "block_summary.csv", block_summary),
    (REPORT_DIR / "before_after_counts.csv", count_report),
    (REPORT_DIR / "block_size_distribution.csv", block_size_distribution),
    (REPORT_DIR / "all_tied_block_events.csv", all_tied_events),
    (REPORT_DIR / "block_examples.csv", examples),
]
for path, frame in exports:
    # Exclusive creation also prevents accidental overwrites when this cell is rerun.
    frame.to_csv(path, index=False, mode="x")
    manifest["outputs"].append({
        "path": str(path), "rows": len(frame), "bytes": path.stat().st_size,
        "fits_excel_worksheet_with_header": bool(len(frame) <= 1_048_575),
    })
for path in [RAW_FILE, PREVIOUS_FILE]:
    assert file_sha256(path) == input_checksums[str(path.relative_to(PROJECT_ROOT))], (
        f"Input changed during this run: {path}"
    )
manifest["elapsed_seconds"] = time.perf_counter() - STARTED
manifest["completed_utc"] = datetime.now(timezone.utc).isoformat()
manifest["status"] = "prepared data; not yet training-ready"
manifest["validation"] = {
    "expected_counts": "passed", "source_row_mapping": "passed",
    "event_and_membership_reconciliation": "passed",
    "grouping_and_shuffle_examples": "passed", "input_data_unchanged": "passed",
}
MANIFEST_FILE.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Preparation completed. This is not a train/test split.")
display(pd.DataFrame(manifest["outputs"]))
print("Manifest:", MANIFEST_FILE)

Preparation completed. This is not a train/test split.


,path,rows,bytes,fits_excel_worksheet_with_header
0,E:\ALL MCP Servers\Recommendation\data\interim...,2755641,181863536,False
1,E:\ALL MCP Servers\Recommendation\data\interim...,2755630,85352989,False
2,E:\ALL MCP Servers\Recommendation\data\interim...,2753015,306712810,False
3,E:\ALL MCP Servers\Recommendation\reports\bask...,2,138,True
4,E:\ALL MCP Servers\Recommendation\reports\bask...,7,104,True
5,E:\ALL MCP Servers\Recommendation\reports\bask...,4769,469782,True
6,E:\ALL MCP Servers\Recommendation\reports\bask...,22,2063,True


Manifest: E:\ALL MCP Servers\Recommendation\reports\basket_prepared_20261013_180506_369663\manifest.json


## Manual Excel checks

Open **block_examples.csv** or **all_tied_block_events.csv**, select the data, and press Ctrl+T.

- Filter column A (block_id). Columns D and C (visitorid and timestamp) must be identical inside the block.
- Event rows stay separate. Item ID is column F. In a same-item tie, different actions can share that item.
- Outside the table, use `=COUNTIF(A:A,A2)` for a block's event-row count. Compare it with column H (event_count).
- To count distinct items manually, copy the filtered block's item IDs to a separate scratch sheet and use Data → Remove Duplicates on that scratch copy only. Compare its count with column I (distinct_item_count).
- For a transaction basket, all event values in column E are transaction; transaction ID in G is shared and nonblank; there are multiple item IDs.
- Column B is the raw CSV source row. The full raw CSV exceeds Excel's row limit; use Power Query or pandas to locate rows beyond that limit.
- In **before_after_counts.csv**, transactions should remain 22,457. Only the 460 extra exact-copy rows disappear from the event total.

## Where we stop

We now have event records, item memberships, and time blocks. No internal order has been assigned to basket items.

The next notebook must set chronological train/validation/test boundaries, define training-only eligibility and the item catalog, and report losses from those decisions. The older strict and whole-timeline 5-core files remain historical exploratory outputs.

Both models will need an explicitly documented way to represent unordered history blocks and evaluate multiple valid target items. This preparation alone does not make the data compatible with unmodified SASRec or TiSASRec.